[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JMasr/ai-tutorials/blob/main/en/terminal-10-90-lab.ipynb)

Author: José Manuel Ramírez Sánchez · https://jmramirez.engineer

# The 10% of the terminal: a lab with four flags

Companion to the post [The 10% of the terminal that solves 90% of the job](https://jmramirez.engineer/writing/terminal-10-90/), which explains each skill with animations. Here you practise them on a real machine: each section hides a flag, and `check()` at the end tells you how many you have.

The story is the post's: it is 2:47, the overnight Whisper training run on the `gpu` server has died with `CUDA out of memory`, and you have to bring it back to life. Keep [the cheat sheet](https://jmramirez.engineer/cheatsheets/terminal-10-90-en.pdf) at hand.

### Where this runs

This notebook is made for **Google Colab** (free, CPU is enough). It installs packages, creates a user called `username` and starts an SSH server that only listens inside Colab's virtual machine, on `127.0.0.1`. That machine is yours, temporary, and is deleted when you disconnect.

Do not run it on your own computer or on a shared server: it creates a user with a known password. If you still want to, use a disposable virtual machine, at your own risk. The notebook is provided as is, without warranty (MIT licence).

### The setup

| In the post | In this lab |
|---|---|
| your laptop | the terminal of this Colab machine (you are `root`, your folder is `~`) |
| the server `gpu`, `10.0.3.7` | user `username` on `127.0.0.1`, port `2222`, in the same machine |
| the forgotten Jupyter kernel | a process that eats one CPU core (Colab has no GPU in the free tier) |

Because the "server" lives in the same machine, its address is `127.0.0.1` and it needs `-p 2222` or `Port 2222`. Everything else is exactly as on a real server.

## 0. Set up the lab

Run this cell (about 30 s). Write down the **fingerprint** it prints: you will need it in section 2.

In [ ]:
# Lab setup. Run it once; running it again starts the lab from scratch with new flags.
# Only in Google Colab: it installs packages, creates the user `username` and starts an SSH server
# that listens on 127.0.0.1 only, inside this disposable virtual machine.
import base64, hashlib, json, os, secrets, shutil, socket, subprocess, sys, textwrap, time
from pathlib import Path

M = {'proj': 'projects', 'dl': 'Downloads', 'notes': 'notes.txt', 'note_text': 'whisper: overnight training on gpu, check it in the morning\n', 'colab_only': "This lab only runs in Google Colab: it installs packages, creates a user and starts an SSH server. Open it with the 'Open in Colab' badge. (In a disposable virtual machine, at your own risk: os.environ['LAB_OUTSIDE_COLAB'] = '1'.)", 'installing': 'Installing openssh-server, tmux and nano (about 30 s)…', 'ready': 'Lab ready.\n  laptop: this Colab machine, your folder {laptop}\n  server "gpu": user {user} on 127.0.0.1, port {port} (only reachable from inside this machine)\n  password of {user}, only for ssh-copy-id: {password}\n  fingerprint of gpu: {fp}\n\nNext: open the terminal with the next cell.', 'dry_oom': 'batch_size: {} -> CUDA out of memory (24 GB GPU). Lower it to 16 in configs/train.yaml.', 'dry_ok': 'config OK (batch_size: {}) · {}', 'oom': 'torch.OutOfMemoryError: CUDA out of memory (batch_size: {}). Is the fixed config on this machine?', 'no_tmux': 'Running outside a tmux session called "train": if the connection drops, this dies with it.\n(The flag only shows up inside `tmux new -s train`.)', 'done': 'training finished while you were away · {}', 'noflag': 'training finished. No flag: run it inside the tmux session "train" and leave with Ctrl-b d while it runs.', 'ghost': 'Something is eating a CPU core on gpu. It stops by itself in {} min.', 'setup_first': 'Run the setup cell first.', 'hints': ['Moving around: fix the config with nano and run train.py --dry-run on the laptop.', 'SSH: bring logs/train.log from gpu to the laptop with scp.', 'tmux: train inside the session "train" and leave with Ctrl-b d while it runs.', 'Monitoring: find the process eating the CPU on gpu, and kill it.'], 'ok': 'done', 'all': 'Four out of four. It is 3:20: time for bed.', 'nolog': 'Right flag, but train.log is not on the laptop: bring it with scp.', 'alive': 'Right name, but the process is still running: end it with kill.'}
USER, PASSWORD, PORT = 'username', 'gpu-lab', 2222
LAPTOP = Path.home()                    # the "laptop": your folder in this Colab machine
PROJ = LAPTOP / M['proj'] / 'whisper'
SERVER = Path('/home') / USER           # the "server" gpu: another user, reached through SSH
STATE = LAPTOP / '.lab-10-90'           # flag hashes (reading them gives nothing away)
TRAIN_PY = '#!/usr/bin/env python3\n"""Whisper training (a simulation for the lab). Reads configs/train.yaml."""\nimport base64, os, subprocess, sys, time\nfrom pathlib import Path\n\nHERE = Path(__file__).resolve().parent\nunhide = lambda s: base64.b64decode(s[::-1]).decode()\ncfg = (HERE / \'configs\' / \'train.yaml\').read_text()\nbatch = int(next(l.split(\':\')[1] for l in cfg.splitlines() if l.strip().startswith(\'batch_size\')))\n\nif \'--dry-run\' in sys.argv:\n    if batch > 16:\n        sys.exit(@DRY_OOM@.format(batch))\n    print(@DRY_OK@.format(batch, unhide(@F1@)))\n    sys.exit()\n\nif batch > 16:\n    print(\'epoch 1/12 · loss 2.31\', flush=True)\n    time.sleep(2)\n    sys.exit(@OOM@.format(batch))\n\ndef session():\n    if not os.environ.get(\'TMUX\'):\n        return None\n    return subprocess.run([\'tmux\', \'display-message\', \'-p\', \'#S\'], capture_output=True, text=True).stdout.strip()\n\ndef attached():\n    out = subprocess.run([\'tmux\', \'list-clients\', \'-t\', \'train\'], capture_output=True, text=True).stdout\n    return bool(out.strip())\n\nname = session()\nif name != \'train\':\n    print(@NO_TMUX@, flush=True)\ndetached = False\nfor epoch in range(1, 13):\n    for _ in range(10):\n        time.sleep(1)\n        if name == \'train\' and not attached():\n            detached = True\n    print(f\'epoch {epoch}/12 · loss {2.31 * 0.82 ** epoch:.2f}\', flush=True)\nmsg = @DONE@.format(unhide(@F3@)) if name == \'train\' and detached else @NOFLAG@\nprint(msg)\nwith open(HERE / \'logs\' / \'last-run.txt\', \'a\') as f:   # every run adds its line\n    f.write(msg + \'\\n\')\n'
TRAIN_YAML = 'model: openai/whisper-small\nlanguage: es\nbatch_size: 32\nlearning_rate: 1.0e-5\n'
OLD_LOG = """[02:41:07] run {run} · model openai/whisper-small · batch_size 32
[02:41:09] epoch 1/12 · loss 2.31
[02:46:52] epoch 2/12 · loss 1.87
[02:47:03] torch.OutOfMemoryError: CUDA out of memory. Tried to allocate 1.20 GiB
[02:47:03] GPU 0 has a total capacity of 23.99 GiB of which 412.00 MiB is free.
[02:47:03] Process 2290 has 9.30 GiB memory in use.
[02:47:03] run {run} · status FAILED · keep this file: {flag}
"""

if 'google.colab' not in sys.modules and os.environ.get('LAB_OUTSIDE_COLAB') != '1':
    raise RuntimeError(M['colab_only'])


def run(cmd, check=True):
    return subprocess.run(cmd, shell=True, check=check, text=True, capture_output=True)


def write(path, text, mode=0o644):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text)
    path.chmod(mode)


sha = lambda s: hashlib.sha256(str(s).strip().encode()).hexdigest()
hide = lambda s: base64.b64encode(s.encode()).decode()[::-1]   # only so that grep does not find the flags


def setup():
    print(M['installing'])
    run('apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq openssh-server tmux nano procps > /dev/null')
    run('service ssh stop', check=False)                     # the default server on port 22 is not needed
    run('pip install -q colab-xterm', check=False)
    run(f'id -u {USER} || useradd -m -s /bin/bash {USER}')
    run(f'echo "{USER}:{PASSWORD}" | chpasswd')
    run(f'pkill -u {USER}', check=False)                     # leftovers of a previous run
    for p in (PROJ.parent, SERVER / 'whisper', SERVER / '.ssh', SERVER / '.local', STATE):
        shutil.rmtree(p, ignore_errors=True)

    flags = {f'FLAG{i}': f'FLAG{{{secrets.token_hex(4)}}}' for i in (1, 2, 3)}
    flags['FLAG4'] = f'ghost-{secrets.token_hex(3)}'
    code = TRAIN_PY.replace('@F1@', repr(hide(flags['FLAG1']))).replace('@F3@', repr(hide(flags['FLAG3'])))
    for k in ('dry_oom', 'dry_ok', 'oom', 'no_tmux', 'done', 'noflag'):
        code = code.replace(f'@{k.upper()}@', repr(M[k]))

    # The laptop: a project with the config that broke the night's training.
    for root in (PROJ, SERVER / 'whisper'):
        write(root / 'train.py', code, 0o755)
        write(root / 'configs' / 'train.yaml', TRAIN_YAML)
        (root / 'data').mkdir(exist_ok=True)
    (LAPTOP / M['dl']).mkdir(exist_ok=True)
    write(LAPTOP / M['notes'], M['note_text'])
    # The server: the log of the failed run, and a prompt that says where you are.
    write(SERVER / 'whisper' / 'logs' / 'train.log', OLD_LOG.format(run=secrets.token_hex(3), flag=flags['FLAG2']))
    for home, host in ((SERVER, 'gpu'), (LAPTOP, 'laptop')):
        rc = home / '.bashrc'
        text = rc.read_text() if rc.exists() else ''
        if '# lab-10-90' not in text:
            rc.write_text(text + f"\n# lab-10-90\nPS1='\\u@{host}:\\w\\$ '\n")
    (SERVER / '.ssh').mkdir(mode=0o700)
    run(f'chown -R {USER}: {SERVER}')
    STATE.mkdir(mode=0o700)
    (STATE / 'flags.json').write_text(json.dumps({k: sha(v) for k, v in flags.items()}))
    (STATE / 'ghost').write_text(flags['FLAG4'])

    # The SSH server of gpu: its own configuration, on 127.0.0.1:2222 only.
    run('mkdir -p /run/sshd && ssh-keygen -A')
    write('/etc/ssh/sshd_lab_config', textwrap.dedent(f"""\
        Port {PORT}
        ListenAddress 127.0.0.1
        HostKey /etc/ssh/ssh_host_ed25519_key
        PidFile /run/sshd-lab.pid
        AllowUsers {USER}
        PermitRootLogin no
        PubkeyAuthentication yes
        PasswordAuthentication yes
        KbdInteractiveAuthentication no
        UsePAM no
        AllowTcpForwarding no
        X11Forwarding no
        Subsystem sftp /usr/lib/openssh/sftp-server
        """))
    run('kill "$(cat /run/sshd-lab.pid)"', check=False)
    time.sleep(0.5)
    run('/usr/sbin/sshd -f /etc/ssh/sshd_lab_config')
    for _ in range(20):
        with socket.socket() as s:
            if s.connect_ex(('127.0.0.1', PORT)) == 0:
                break
        time.sleep(0.25)
    fp = run('ssh-keygen -lf /etc/ssh/ssh_host_ed25519_key.pub').stdout.split()[1]
    print(M['ready'].format(laptop=LAPTOP, user=USER, port=PORT, password=PASSWORD, fp=fp))


def ghost(minutes=10):
    """Start the process that eats the CPU on gpu: one core, lowest priority, gone after `minutes`."""
    if not (STATE / 'ghost').exists():
        raise RuntimeError(M['setup_first'])
    name = (STATE / 'ghost').read_text().strip()
    binary = SERVER / '.local' / 'bin' / name
    binary.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(shutil.which('yes'), binary)
    run(f'chown -R {USER}: {SERVER / ".local"}')
    subprocess.Popen(['runuser', '-u', USER, '--', 'sh', '-c',
                      f'nice -n 19 "{binary}" > /dev/null & g=$!; sleep {minutes * 60}; kill $g 2>/dev/null'],
                     start_new_session=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print(M['ghost'].format(minutes))


def check(*answers):
    """Check the flags you found, in any order: check('FLAG{...}', 'ghost-...')."""
    if not (STATE / 'flags.json').exists():
        raise RuntimeError(M['setup_first'])
    hashes = json.loads((STATE / 'flags.json').read_text())
    found = {k for a in answers for k, h in hashes.items() if sha(a) == h}
    notes = dict(zip(sorted(hashes), M['hints']))
    if 'FLAG2' in found and not (list(PROJ.rglob('train.log')) or (LAPTOP / 'train.log').exists()):
        found.discard('FLAG2'); notes['FLAG2'] = M['nolog']
    name = (STATE / 'ghost').read_text().strip()
    if 'FLAG4' in found and run(f'pgrep -x {name}', check=False).returncode == 0:
        found.discard('FLAG4'); notes['FLAG4'] = M['alive']
    for i, k in enumerate(sorted(hashes), 1):
        print(f'{"✓" if k in found else "·"} {i}. {M["ok"] if k in found else notes[k]}')
    if len(found) == 4:
        print('\n' + M['all'])


setup()

## Open the terminal

The next cell opens a terminal below it, with [colab-xterm](https://github.com/InfuseAI/colab-xterm). Everything from here on is typed **there**, not in cells. The terminal stays open while you run other cells.

If it does not open, Colab Pro has its own **Terminal** button at the bottom left.

In [ ]:
%load_ext colabxterm
%xterm

## 1. Moving around without a GUI

First, on the laptop: the config of the night's training has `batch_size: 32`, which does not fit on the GPU.

1. Go to your home folder, check where you are and list what is there.
2. Go into `projects/whisper` and list it.
3. Run `python3 train.py --dry-run`: it checks the config without training. It fails.
4. Open `configs/train.yaml` with `nano`, change `32` to `16`, save and exit.
5. Run the dry run again. **Flag 1** is in its output.

<details><summary>Hint</summary>

```bash
cd ~
pwd
ls
cd projects/whisper
ls
python3 train.py --dry-run
nano configs/train.yaml      # Ctrl-O and Enter save, Ctrl-X exits
python3 train.py --dry-run
```
</details>

## 2. Getting into the server

The config is fixed on the laptop, but the training lives on `gpu`.

1. Create your key pair with `ssh-keygen -t ed25519`. Press Enter to accept the path; the passphrase is optional in this lab.
2. Copy the public key to the server with `ssh-copy-id -p 2222 username@127.0.0.1`. The first time it shows the server's fingerprint: answer `yes` only if it matches the one the setup cell printed. The password is `gpu-lab`.
3. Give the server a short name in `~/.ssh/config`:

```
Host gpu
    HostName 127.0.0.1
    Port 2222
    User username
```

4. Get in with `ssh gpu`. The prompt changes to `username@gpu`. Have a look around with `ls`, and leave with `exit`.
5. Back in `~/projects/whisper` on the laptop, send the fixed config to the server, into `whisper/configs/`.
6. Bring the log of the failed run, `whisper/logs/train.log`, from the server to the laptop. **Flag 2** is on its last line. `check()` also wants the file on the laptop.

<details><summary>Hint</summary>

```bash
ssh-keygen -t ed25519
ssh-copy-id -p 2222 username@127.0.0.1
nano ~/.ssh/config
ssh gpu
exit
cd ~/projects/whisper
scp configs/train.yaml gpu:whisper/configs/
scp gpu:whisper/logs/train.log .
nano train.log
```
</details>

## 3. Keeping the training running: tmux

Now relaunch the training on `gpu`, in a way that survives you leaving. It takes 2 minutes.

1. Get into `gpu` and go into `whisper`.
2. Open a tmux session called `train` and launch `python3 train.py` inside it. If it fails with `CUDA out of memory`, the fixed config is not on the server yet (section 2).
3. Leave the session with `Ctrl-b d` while it trains, and leave the server with `exit`.
4. Wait about 2 minutes. Then get back into `gpu` and attach to the session. **Flag 3** is on the last line (it is also saved in `whisper/logs/last-run.txt`).

The flag only shows up if the training ran inside the `train` session and you left it while it was running.

<details><summary>Hint</summary>

```bash
ssh gpu
cd whisper
tmux new -s train
python3 train.py
# Ctrl-b, let go, d
exit
# ...2 minutes later
ssh gpu
tmux ls
tmux attach -t train
```
</details>

## 4. Seeing what is going on: top

The training is running, but something else is eating the server. Run the next cell: it starts a process on `gpu`, as user `username`, that keeps one CPU core busy (it stops by itself after 10 minutes).

In [ ]:
ghost()

In the terminal:

1. Get into `gpu` and open `top`. Find the process at the top of `%CPU` and write down its PID and its name (column `COMMAND`). Leave with `q`.
2. Before killing it, check whose it is and what it is with `ps -o user=,args= -p PID`.
3. It is yours (`username`): end it with `kill PID`. Run `top` again to check that it is gone.
4. **Flag 4** is the name of the process, `ghost-` followed by six characters.

With a GPU runtime in Colab you can also run `nvidia-smi` in the laptop terminal and see a real one.

<details><summary>Hint</summary>

```bash
ssh gpu
top                        # q to quit
ps -o user=,args= -p PID   # the PID you saw in top
kill PID
top
```
</details>

## Check your flags

Paste the four flags into `check()`, in any order, and run the cell. It tells you which are right and what is missing.

In [ ]:
check('FLAG{...}', 'FLAG{...}', 'FLAG{...}', 'ghost-...')

## And now

You have done each skill once from start to finish. Do it again tomorrow without the hints, and it will start coming from memory.

- The post with the animations: [The 10% of the terminal that solves 90% of the job](https://jmramirez.engineer/writing/terminal-10-90/)
- [The one-page cheat sheet](https://jmramirez.engineer/cheatsheets/terminal-10-90-en.pdf)
- To start again from scratch, with new flags, run the setup cell again. To delete everything, *Runtime → Disconnect and delete runtime*.